In [1]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, median, trim, initcap, upper, round, year, concat_ws
from pyspark.sql.types import DateType, DoubleType, IntegerType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("TradeCorpETL")

spark = SparkSession.builder.appName("TradeCorpETL").getOrCreate()

logger.info(f"Session Spark créée - version {spark.version}")
logger.info("Chargement des fichiers CSV")

df_customers = spark.read.csv("../data/customers.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("../data/orders.csv", header=True, inferSchema=True)
df_order_details = spark.read.csv("../data/order_details.csv", header=True, inferSchema=True)
df_products = spark.read.csv("../data/products.csv", header=True, inferSchema=True)
df_categories = spark.read.csv("../data/categories.csv", header=True, inferSchema=True)
df_suppliers = spark.read.csv("../data/suppliers.csv", header=True, inferSchema=True)
df_employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
df_shippers = spark.read.csv("../data/shippers.csv", header=True, inferSchema=True)

logger.info("Chargement terminé : les 8 DataFrames sont disponibles")

dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers
}

2026-09-01 09:50:31,910 | INFO | NumExpr defaulting to 12 threads.
2026-09-01 09:50:35,733 | INFO | Session Spark créée - version 4.2.0
2026-09-01 09:50:35,734 | INFO | Chargement des fichiers CSV
2026-09-01 09:50:40,038 | INFO | Chargement terminé : les 8 DataFrames sont disponibles


**Question 11** : Pour chaque DataFrame, compter le nombre de valeurs nulles par colonne. Utiliser une boucle et
df.filter(col(c).isNull()).count().

In [2]:
logger.info("Question 11 - Compter les valeurs nulles")

for name, df in dataframes.items():
    logger.info(f"Analyse des valeurs nulles : {name}")
    print(f"\n{name} :")

    for c in df.columns:
        nb_nulls = df.filter(col(c).isNull()).count()
        print(f"{c}: {nb_nulls}")

logger.info("Question 11 terminée")

2026-09-01 09:50:40,048 | INFO | Question 11 - Compter les valeurs nulles
2026-09-01 09:50:40,050 | INFO | Analyse des valeurs nulles : customers



customers :
customer_id: 0
company_name: 0
contact_name: 0
contact_title: 0
address: 0
city: 0
region: 60
postal_code: 1
country: 0
phone: 0


2026-09-01 09:50:41,857 | INFO | Analyse des valeurs nulles : orders


fax: 22

orders :
order_id: 0
customer_id: 0
employee_id: 0
order_date: 0
required_date: 0
shipped_date: 21
ship_via: 0
freight: 0
ship_name: 0
ship_address: 0
ship_city: 0
ship_region: 507


2026-09-01 09:50:43,323 | INFO | Analyse des valeurs nulles : order_details


...

homepage: 24

employees :
employee_id: 0
last_name: 0
first_name: 0
title: 0
title_of_courtesy: 0
birth_date: 0
hire_date: 0
address: 0
city: 0
region: 4
postal_code: 0
country: 0
home_phone: 0
extension: 0
photo: 9
notes: 0
reports_to: 1


2026-09-01 09:50:47,506 | INFO | Analyse des valeurs nulles : shippers


photo_path: 0

shippers :
shipper_id: 0
company_name: 0


2026-09-01 09:50:47,743 | INFO | Question 11 terminée


phone: 0


**Question 12** : Dans df_orders, supprimer les lignes où shipped_date est null (commandes non livrées). Dans df_products,
remplacer les valeurs nulles de unit_price par la médiane.

In [3]:
logger.info("Question 12 - Traitement des valeurs nulles")

nb_orders_avant = df_orders.count()
df_orders = df_orders.filter(col("shipped_date").isNotNull())
nb_orders_apres = df_orders.count()

logger.info(
    f"Commandes sans shipped_date supprimées : "
    f"{nb_orders_avant - nb_orders_apres}"
)

mediane_unit_price = df_products.agg(median("unit_price").alias("mediane")).first()["mediane"]
logger.info(f"Médiane de unit_price calculée : {mediane_unit_price}")

df_products = df_products.fillna({"unit_price": mediane_unit_price})

logger.info("Valeurs nulles de unit_price remplacées par la médiane")

nb_nulls = df_products.filter(col("unit_price").isNull()).count()

logger.info(f"Valeurs nulles restantes dans unit_price : {nb_nulls}")
logger.info("Question 12 terminée")

2026-09-01 09:50:47,749 | INFO | Question 12 - Traitement des valeurs nulles
2026-09-01 09:50:47,963 | INFO | Commandes sans shipped_date supprimées : 21
2026-09-01 09:50:48,233 | INFO | Médiane de unit_price calculée : 19.5
2026-09-01 09:50:48,262 | INFO | Valeurs nulles de unit_price remplacées par la médiane
2026-09-01 09:50:48,339 | INFO | Valeurs nulles restantes dans unit_price : 0
2026-09-01 09:50:48,339 | INFO | Question 12 terminée


**Question 13** : Dans df_orders, caster order_date, required_date et shipped_date en type DateType. Dans df_order_details,
caster unit_price en DoubleType et quantity en IntegerType.

In [4]:
logger.info("Question 13 - Conversion des types de colonnes")

df_orders = (df_orders
             .withColumn("order_date", col("order_date").cast(DateType()))
             .withColumn("required_date", col("required_date").cast(DateType()))
             .withColumn("shipped_date", col("shipped_date").cast(DateType()))
            )

logger.info("Dates de df_orders converties en DateType")

df_order_details = (df_order_details
                    .withColumn("unit_price", col("unit_price").cast(DoubleType()))
                    .withColumn("quantity", col("quantity").cast(IntegerType()))
                   )

logger.info("unit_price converti en DoubleType et quantity en IntegerType")

logger.info("Question 13 terminée")

2026-09-01 09:50:48,346 | INFO | Question 13 - Conversion des types de colonnes
2026-09-01 09:50:48,408 | INFO | Dates de df_orders converties en DateType
2026-09-01 09:50:48,433 | INFO | unit_price converti en DoubleType et quantity en IntegerType
2026-09-01 09:50:48,434 | INFO | Question 13 terminée


In [5]:
df_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)



In [6]:
df_order_details.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)



**Question 14** : Dans df_customers, appliquer TRIM sur toutes les colonnes texte. Mettre contact_name en title case avec
initcap(). Mettre country en majuscules avec upper().

In [7]:
logger.info("Question 14 - Normalisation des clients")

df_customers = df_customers.select(*[
    initcap(trim(col(c_name))).alias(c_name)
    if c_name == "contact_name"
    
    else upper(trim(col(c_name))).alias(c_name)
    if c_name == "country"
    
    else trim(col(c_name)).alias(c_name)
    if c_type == "string"
    
    else col(c_name)

    for c_name, c_type in df_customers.dtypes
])
logger.info("TRIM appliqué aux colonnes texte")
logger.info("contact_name est en title case et country est en majuscules")

df_customers.select("contact_name","country").show(10)

logger.info("Question 14 terminée")

2026-09-01 09:50:48,498 | INFO | Question 14 - Normalisation des clients
2026-09-01 09:50:48,562 | INFO | TRIM appliqué aux colonnes texte
2026-09-01 09:50:48,562 | INFO | contact_name est en title case et country est en majuscules
2026-09-01 09:50:49,751 | INFO | Question 14 terminée


+------------------+-------+
|      contact_name|country|
+------------------+-------+
|      Maria Anders|GERMANY|
|      Ana Trujillo| MEXICO|
|    Antonio Moreno| MEXICO|
|      Thomas Hardy|     UK|
|Christina Berglund| SWEDEN|
|        Hanna Moos|GERMANY|
|Frédérique Citeaux| FRANCE|
|     Martín Sommer|  SPAIN|
|  Laurence Lebihan| FRANCE|
| Elizabeth Lincoln| CANADA|
+------------------+-------+
only showing top 10 rows


**Question 15** : Dans df_order_details, renommer unit_price en prix_unitaire et quantity en quantite. Dans df_orders, renommer
ship_via en shipper_id

In [8]:
logger.info("Question 15 - Renommage des colonnes")

df_order_details = df_order_details.withColumnRenamed("unit_price", "prix_unitaire")
df_order_details = df_order_details.withColumnRenamed("quantity", "quantite")
df_orders = df_orders.withColumnRenamed("ship_via", "shipper_id")


logger.info("Colonnes renommées : unit_price en prix_unitaire, quantity en quantite, ship_via en shipper_id")

logger.info("Question 15 terminée")

2026-09-01 09:50:49,758 | INFO | Question 15 - Renommage des colonnes
2026-09-01 09:50:49,781 | INFO | Colonnes renommées : unit_price en prix_unitaire, quantity en quantite, ship_via en shipper_id
2026-09-01 09:50:49,782 | INFO | Question 15 terminée


**Question 16** : Dans df_order_details, ajouter une colonne sous_total = prix_unitaire * quantite * (1 - discount). Arrondir à 2
décimales avec round().

In [10]:
logger.info("Question 16 - Calcul de la colonne sous_total")

df_order_details = df_order_details.withColumn(
    "sous_total",
    round(col("prix_unitaire")* col("quantite")* (1 - col("discount")),2)
)

df_order_details.show()

logger.info("Colonne sous_total ajoutée et arrondie à 2 décimales")
logger.info("Question 16 terminée")

2026-09-01 09:51:00,948 | INFO | Question 16 - Calcul de la colonne sous_total
2026-09-01 09:51:01,088 | INFO | Colonne sous_total ajoutée et arrondie à 2 décimales
2026-09-01 09:51:01,088 | INFO | Question 16 terminée


+--------+----------+-------------+--------+--------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|
+--------+----------+-------------+--------+--------+----------+
|   10248|        11|         14.0|      12|     0.0|     168.0|
|   10248|        42|          9.8|      10|     0.0|      98.0|
|   10248|        72|         34.8|       5|     0.0|     174.0|
|   10249|        14|         18.6|       9|     0.0|     167.4|
|   10249|        51|         42.4|      40|     0.0|    1696.0|
|   10250|        41|          7.7|      10|     0.0|      77.0|
|   10250|        51|         42.4|      35|    0.15|    1261.4|
|   10250|        65|         16.8|      15|    0.15|     214.2|
|   10251|        22|         16.8|       6|    0.05|     95.76|
|   10251|        57|         15.6|      15|    0.05|     222.3|
|   10251|        65|         16.8|      20|     0.0|     336.0|
|   10252|        20|         64.8|      40|    0.05|    2462.4|
|   10252|        33|    

**Question 17** : Dans df_products, ajouter une colonne en_stock (True si units_in_stock > 0). Dans df_orders, ajouter une
colonne is_shipped (True si shipped_date n'est pas null)

In [ ]:
logger.info("Question 17 - Création des indicateurs")

df_products = df_products.withColumn("en_stock", col("units_in_stock") > 0)
logger.info("Colonne en_stock ajoutée")

df_orders = df_orders.withColumn("is_shipped", col("shipped_date").isNotNull())
logger.info("Colonne is_shipped ajoutée")
logger.info("Question 17 terminée")

**Question 18** : Vérifier s'il y a des doublons dans df_customers sur customer_id. Utiliser distinct() et count() pour comparer.
Supprimer les doublons si nécessaire

In [ ]:
logger.info("Question 18 - Vérification des doublons clients")

nb_customers = df_customers.count()

nb_customers_distinct = (df_customers.select("customer_id").distinct().count())

print(f"Nombre de lignes : {nb_customers}")
print(f"Nombre de customer_id distincts : {nb_customers_distinct}")

if nb_customers != nb_customers_distinct:
    logger.warning("Doublons détectés sur customer_id")
    df_customers = df_customers.dropDuplicates(["customer_id"])
    logger.info("Doublons supprimés")
else:
    logger.info("Aucun doublon détecté sur customer_id")

logger.info("Question 18 terminée")

**Question 19** : Filtrer df_orders pour ne garder que les commandes de 1997. Filtrer df_products pour ne garder que les produits
en stock (units_in_stock > 0) et non discontinués.

In [ ]:
logger.info("Question 19 - Filtrage des données")

nb_orders_avant = df_orders.count()
df_orders = df_orders.filter(year(col("order_date")) == 1997)
nb_orders_apres = df_orders.count()

logger.info(
    f"Commandes : {nb_orders_avant} -> {nb_orders_apres} lignes "
    f"({nb_orders_avant - nb_orders_apres} supprimées)"
)

nb_products_avant = df_products.count()
df_products = df_products.filter((col("en_stock") == True) & (col("discontinued") == 0))
nb_products_apres = df_products.count()

logger.info(
    f"Produits : {nb_products_avant} -> {nb_products_apres} lignes "
    f"({nb_products_avant - nb_products_apres} supprimées)"
)  

logger.info("Question 19 terminée")

**Question 20** : Dans df_employees, sélectionner uniquement : employee_id, first_name, last_name, title, hire_date, city,
country. Créer une colonne full_name = first_name + ' ' + last_name.

In [ ]:
logger.info("Question 20 - Selection et transformation des employés")

df_employees = df_employees.select("employee_id", "first_name", "last_name", "title", "hire_date", "city", "country",
                                  concat_ws(" ", col("first_name"), col("last_name")).alias("full_name"))
logger.info("Colonnes sélectionnées et colonne full_name créée")
logger.info("Question 20 terminée")

Exportation en parquet des DF

In [ ]:
logger.info("Début de l'exportation en parquet")
dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers
}

logger.info("Export des DataFrames au format Parquet")

for name, df in dataframes.items():
    logger.info(f"Export de {name}")
    df.write.mode("overwrite").parquet(f"../data/tmp/{name}")
    logger.info(f"Export de {name} terminé")

logger.info("Export des DataFrames terminé")

In [ ]:
df = spark.read.parquet("../data/tmp/employees")
df.show(20)